In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [2]:
subs_df['original_comid'] = subs_df.index

# A small % of gauges are dropped if they don't match well.
common_idx = subs_df.index.intersection(gauges.index)
subs_df.loc[common_idx, 'original_comid'] = gauges.loc[common_idx, 'COMID']

subs_df

,is_gauge,uparea_km2,outlet_id,original_comid
site_id,,,,
23021321,False,209.123678,EAUF-A3750050,23021321
23021261,False,213.109547,EAUF-A3750050,23021261
23021244,False,291.030672,EAUF-A3750050,23021244
23021122,False,477.964586,EAUF-A3750050,23021122
23021112,False,394.295017,EAUF-A3750050,23021112
...,...,...,...,...
81003765,False,1613.152478,USGS-15908000,81003765
81001414,False,1473.949940,USGS-15908000,81001414
81001413,False,2751.705206,USGS-15908000,81001413


In [3]:
datasets = Path("/nas/cee-ice/data")
glow_dir = datasets / "GLOW-S" / "daily_reach_aggregated"

glow_dfs = []
for region_file in glow_dir.glob('*_daily_median.parquet'):
    glow_dfs.append(pd.read_parquet(region_file))
                    
glow = pd.concat(glow_dfs)

In [13]:
batch_subs

['56155034',
 '56154994',
 '56139517',
 '56155445',
 '56143960',
 '56143934',
 '56143416',
 '56142915',
 '56142244',
 '56141694',
 '56141508',
 '56141344',
 '56141308',
 '56141201',
 '56139780',
 '56154829',
 'ABOM-100288010',
 'ABOM-97444010',
 'ABOM-102903010',
 'ABOM-102984010']

In [15]:
subs_df.loc[subbasin]

is_gauge                   False
uparea_km2             702.42016
outlet_id         ABOM-100288010
original_comid          56155034
Name: 56155034, dtype: object

In [16]:
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-zbs_vD")
store = ZarrBasinStore(store_path)


def get_glow_s(comid):
    comid = int(comid)
    try:
        glow_ix = glow.xs(comid, level='COMID')
    except KeyError:
        return None
    
    glow_ix.index = pd.to_datetime(glow_ix.index).tz_localize('UTC')
    return glow_ix.rename(columns={'width':'glow_width'})

    

BATCH_SIZE = 1000
batches = []
for basin_id, basin_group in subs_df.groupby('outlet_id'):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)

    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        batches.append((basin_id, batch_subs))

        
# Iterate over basins (distinct Zarr stores)
for basin_id, batch_subs in tqdm(batches):
    basin_subs = basin_subbasin_dict[basin_id]
    batch_df_list = []
    for subbasin in batch_subs:
        comid = subs_df.loc[subbasin]['original_comid']
        site_df = get_glow_s(comid)
            
        if site_df is not None and not site_df.empty:
            site_df['subbasin'] = subbasin
            batch_df_list.append(site_df)

    # If the whole batch is empty, we technically don't need to write anything 
    # since the zarr was initialized with NaNs/empty).
    if batch_df_list:
        batch_df = pd.concat(batch_df_list)
        store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs)

100%|██████████| 1508/1508 [38:04<00:00,  1.52s/it]  


In [ ]:
store.compute_and_store_stats()